# Taller N.º 11 — Spark Structured Streaming: ventanas, watermark y checkpoints

**Sesiones 38 y 39 · Big Data 2026-II · Universidad Externado de Colombia**

**Carácter del taller:** calificable. Structured Streaming no entra en ningún examen, así que el producto de este taller forma parte directa de la nota del componente de talleres. El formato es el de los Talleres 7, 8 y 10: ejecución con inteligencia artificial permitida (Fase 1), corte, cuestionario sin inteligencia artificial (Fase 2) y evidencia en capturas.

**El flujo del taller**

productor en Python → topic `t11-lecturas` (Kafka) → consultas de Structured Streaming (Spark sobre YARN) → tablas en memoria (Ejecución 2) y archivos Parquet en HDFS (Ejecución 3)

**Qué cambia respecto del Taller 10**

1. **Vuelve el portal.** La evidencia está en la pestaña **Structured Streaming** de la Spark UI, y la ejecución propia la acredita de nuevo el identificador `application_…` de la barra de direcciones, como en los Talleres 7 y 8.
2. **Se necesitan HDFS, YARN y el History Server**, además de Kafka: Spark corre sobre YARN y los checkpoints se guardan en HDFS.
3. **No hay terminales.** El productor de eventos es un programa en Python que la propia celda lanza como un proceso aparte del sistema operativo. La celda muestra su avance y, al final, un resumen.
4. **Durante la Ejecución 2 corren dos consultas a la vez** sobre el mismo topic.

**Lo que este taller no usa.** El modo de salida Update y la salida a consola se vieron en la demostración. La salida a consola imprime en la terminal donde se inició Jupyter, no en el notebook; por eso aquí se usan tablas en memoria y archivos.

---
## Antes de empezar (1): las cuatro capturas obligatorias

Se exigen únicamente estas cuatro. Una captura adicional no otorga puntos, y la ausencia de una obligatoria deja sin respaldo las preguntas que dependan de ella.

| N.º | Ejecución | Qué debe verse |
|---|---|---|
| **C1** | E2 | Salida de la celda E2: el bloque **RESUMEN** completo, desde su encabezado hasta la línea que explica el símbolo `(*)` |
| **C2** | E2 | Página de estadísticas de la consulta `e2_completo`: los gráficos **Aggregated Number Of Total State Rows** y **Aggregated Number Of Rows Dropped By Watermark**, con el recuadro del **último punto** de *Total State Rows* visible |
| **C3** | E2 | Página de estadísticas de la consulta `e2_anexar`: desde **Global Watermark Gap** hasta **Aggregated Number Of Rows Dropped By Watermark**, con el recuadro del **punto más alto** de *Rows Dropped By Watermark* visible |
| **C4** | E3b | Página principal de la pestaña Structured Streaming: la tabla **Completed Streaming Queries** con sus cuatro filas |

**Criterio de suficiencia**

- En C1 debe verse el encabezado del bloque: fecha, hora y Application ID.
- En C2, C3 y C4 debe verse la **barra de direcciones** del navegador, con el identificador `application_…`. En C2 y C3, la dirección termina en `id=` seguido del RunId de la consulta: debe verse al menos su comienzo.
- El **recuadro** aparece al pasar el cursor sobre un punto de la línea del gráfico: muestra el valor y la hora de ese lote. La captura se toma con el cursor encima, para que el recuadro quede visible.
- Si los gráficos pedidos no caben en la pantalla, reducir el zoom del navegador con `Ctrl` + `−` hasta que quepan.
- Cada captura debe presentar la fecha y hora del sistema.
- Cada imagen se relaciona en su subsección C1, C2, C3 o C4 del archivo de Word, en ese orden.

---
## Antes de empezar (2): el portal, el notebook y las celdas que no se repiten

**Cómo llegar a la pestaña Structured Streaming.** En el portal de YARN (`http://192.168.56.101:8088`), abrir la aplicación `Taller11 - Structured Streaming`, que aparece en estado `RUNNING`, y seguir el enlace **ApplicationMaster** de la columna *Tracking UI*. La pestaña **Structured Streaming** no existe al lanzar la aplicación: aparece cuando la celda E2 inicia la primera consulta. Si el portal ya estaba abierto, recargar la página (F5) para verla.

- La página principal de la pestaña tiene dos tablas: **Active Streaming Queries**, con las consultas en curso, y **Completed Streaming Queries**, con las que ya se detuvieron.
- Al hacer clic en el **Run ID** de una consulta se abre su página de estadísticas, con un gráfico por métrica y por lote.
- Las consultas detenidas siguen consultables mientras la aplicación de Spark esté viva.

**No usar «Run All» (ejecutar todas las celdas).** Las celdas deben ejecutarse una por una, en orden.

**No ejecutar `spark.stop()` ni usar *Shut Down Kernel*.** La aplicación debe seguir viva durante la Fase 2. Cerrar la pestaña del notebook no la detiene.

**Repetir celdas.** Los parámetros, las funciones, la preparación (P0 a P3) y E1 pueden ejecutarse de nuevo sin consecuencias. E2, E3a y E3b no se repiten: la segunda vez solo muestran el resumen que guardaron, porque repetirlas cambiaría las cifras del taller.

---
## Preparación (al inicio de la sesión, durante el control de lectura)

1. Encender las tres máquinas virtuales.
2. Iniciar HDFS, YARN y el History Server de Spark, como en los Talleres 7 y 8.
3. Iniciar Jupyter y abrir este notebook en el maestro.
4. Ejecutar, en orden, las celdas de **parámetros** y **funciones**, y después **P0, P1, P2, P3 y E1**.

Algunas celdas tardan uno o dos minutos. Mientras tanto se responde el control de lectura en Wayground, en el navegador del equipo anfitrión.

In [1]:
# Parametros del taller. No modificar durante la sesion.
import os, sys, re, json, time, random, datetime, subprocess

os.environ["JAVA_HOME"] = "/opt/hadoop/jdk"
os.environ["PATH"] = "/opt/hadoop/jdk/bin:" + os.environ["PATH"]

KB  = "/opt/kafka/bin"                                                # herramientas de consola de Kafka
BS  = "192.168.56.101:9092,192.168.56.102:9092,192.168.56.103:9092"   # brokers de arranque
T11 = "/home/hadoop/t11"                                              # carpeta local del taller (maestro)
HDFS_T11 = "hdfs://hadoop-virtualbox:9000/user/hadoop/t11"            # carpeta del taller en HDFS

TOPIC       = "t11-lecturas"
PARTICIONES = 3
REPLICACION = 2

# Structured Streaming
TRIGGER           = "5 seconds"     # cada cuanto se lanza un micro-batch
VENTANA           = "30 seconds"    # duracion de cada ventana
DESLIZAMIENTO     = "10 seconds"    # desplazamiento de las ventanas sliding (E3)
RETRASO_WATERMARK = "10 seconds"    # umbral del watermark
PARTICIONES_SHUFFLE = 4             # particiones del estado; no se cambia despues de crear un checkpoint

# Recursos de la aplicacion
MEMORIA_EXECUTOR    = "1g"
CORES_EXECUTOR      = "2"
INSTANCIAS_EXECUTOR = "2"

# Rutas derivadas
JARS_DIR      = T11 + "/jars"
PRODUCTOR     = T11 + "/productor.py"
SALIDA_E3     = HDFS_T11 + "/salida/e3_ventanas"
CHECKPOINT_E3 = HDFS_T11 + "/checkpoints/e3_ventanas"

# Conector de Kafka para Spark 3.5.7 (Scala 2.12) y sus dependencias: (ruta en Maven, artefacto, version)
ARTEFACTOS = [
    ("org/apache/spark",   "spark-sql-kafka-0-10_2.12",            "3.5.7"),
    ("org/apache/spark",   "spark-token-provider-kafka-0-10_2.12", "3.5.7"),
    ("org/apache/kafka",   "kafka-clients",                        "3.4.1"),
    ("org/apache/commons", "commons-pool2",                        "2.11.1"),
]
JARS = ["{}/{}-{}.jar".format(JARS_DIR, a, v) for _, a, v in ARTEFACTOS]

os.makedirs(JARS_DIR, exist_ok=True)
print("Parametros cargados.")

Parametros cargados.


In [2]:
# Funciones del taller. No es necesario leerlas; las ejecuciones las usan.

# ---------------------------------------------------------------- identidad y marcas
def cluster_id():
    """cluster.id del cluster Kafka propio."""
    try:
        with open("/opt/kafka/kafka-logs/meta.properties") as f:
            for linea in f:
                if linea.startswith("cluster.id="):
                    return linea.strip().split("=", 1)[1]
    except OSError:
        pass
    return "NO DISPONIBLE"


def app_id():
    try:
        return spark.sparkContext.applicationId
    except NameError:
        return "sin sesion de Spark"


def linea_encabezado(etiqueta):
    return "{} | {:%Y-%m-%d %H:%M:%S} | {} | ClusterId: {}".format(
        etiqueta, datetime.datetime.now(), app_id(), cluster_id())


def encabezado(etiqueta):
    print("=" * 78)
    print(linea_encabezado(etiqueta))
    print("=" * 78)


def ya_ejecutada(etiqueta):
    """E2, E3a y E3b no se repiten: la segunda vez solo muestran el resumen guardado."""
    marca = os.path.join(T11, etiqueta + ".hecho")
    if os.path.exists(marca):
        with open(marca) as f:
            print("ATENCION: {} ya se ejecuto en este cluster ({}).".format(etiqueta, f.read().strip()))
        print("No se repite. Este es el resumen que se guardo entonces:")
        print()
        ruta = os.path.join(T11, etiqueta + "_resumen.txt")
        if os.path.exists(ruta):
            with open(ruta) as f:
                print(f.read())
        return True
    return False


def marcar(etiqueta):
    with open(os.path.join(T11, etiqueta + ".hecho"), "w") as f:
        f.write("{:%Y-%m-%d %H:%M:%S}".format(datetime.datetime.now()))


def habilitar_repeticion(etiqueta):
    """Solo con autorizacion de la docente."""
    marca = os.path.join(T11, etiqueta + ".hecho")
    if os.path.exists(marca):
        os.remove(marca)
        print("Marca de {} eliminada: la ejecucion puede repetirse.".format(etiqueta))


# ---------------------------------------------------------------- HDFS (a traves de Spark)
def _ruta(ruta):
    jvm = spark.sparkContext._jvm
    p = jvm.org.apache.hadoop.fs.Path(ruta)
    return p, p.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())


def existe(ruta):
    p, fs = _ruta(ruta)
    return fs.exists(p)


def borrar(ruta):
    p, fs = _ruta(ruta)
    return fs.delete(p, True) if fs.exists(p) else False


def listar(ruta):
    p, fs = _ruta(ruta)
    if not fs.exists(p):
        return []
    return sorted(s.getPath().getName() for s in fs.listStatus(p) if not s.getPath().getName().startswith("."))


def lotes_en(ruta):
    """Numeros de lote presentes en offsets/ o commits/ de un checkpoint."""
    return sorted(int(n) for n in listar(ruta) if n.isdigit())


# ---------------------------------------------------------------- definicion de las consultas
from pyspark.sql import functions as F, types as T

ESQUEMA = T.StructType([
    T.StructField("estacion", T.StringType()),
    T.StructField("ts_evento_ms", T.LongType()),
    T.StructField("pm25", T.DoubleType()),
])


def leer_kafka():
    """Fuente: el topic del taller, desde el final (solo los mensajes que lleguen despues de arrancar)."""
    return (spark.readStream.format("kafka")
            .option("kafka.bootstrap.servers", BS)
            .option("subscribe", TOPIC)
            .option("startingOffsets", "latest")
            .load())


def eventos(fuente):
    """El value de Kafka llega en binario: se convierte a texto, se interpreta el JSON y se define el tiempo de evento."""
    return (fuente
            .select(F.from_json(F.col("value").cast("string"), ESQUEMA).alias("e"))
            .select("e.*")
            .withColumn("ts_evento", F.expr("timestamp_millis(ts_evento_ms)")))


def conteo_por_ventana(lecturas, deslizamiento=None):
    """Lecturas y PM2.5 medio por ventana y estacion, con watermark sobre el tiempo de evento."""
    if deslizamiento is None:
        ventana = F.window("ts_evento", VENTANA)                    # tumbling
    else:
        ventana = F.window("ts_evento", VENTANA, deslizamiento)     # sliding
    return (lecturas
            .withWatermark("ts_evento", RETRASO_WATERMARK)
            .groupBy(ventana.alias("ventana"), "estacion")
            .agg(F.count("*").alias("lecturas"), F.round(F.avg("pm25"), 1).alias("pm25_medio")))


def arrancar(df, nombre, modo, formato="memory", ruta=None, checkpoint=None):
    for q in spark.streams.active:                                  # restos de un intento interrumpido
        if q.name == nombre:
            q.stop()
    escritor = (df.writeStream
                .queryName(nombre)
                .outputMode(modo)
                .format(formato)
                .option("checkpointLocation", checkpoint or HDFS_T11 + "/checkpoints/" + nombre)
                .trigger(processingTime=TRIGGER))
    if ruta:
        escritor = escritor.option("path", ruta)
    q = escritor.start()
    print("Consulta {} iniciada (modo {}, salida {}).".format(nombre, modo, formato))
    return q


def consulta_e3():
    """Ventanas sliding con watermark, escritas en modo append como archivos Parquet en HDFS."""
    ventanas = (conteo_por_ventana(eventos(leer_kafka()), DESLIZAMIENTO)
                .select(F.col("ventana.start").alias("inicio"), F.col("ventana.end").alias("fin"),
                        "estacion", "lecturas", "pm25_medio"))
    return arrancar(ventanas, "e3_ventanas", "append", formato="parquet", ruta=SALIDA_E3, checkpoint=CHECKPOINT_E3)


def detener(consultas):
    for q in consultas:
        if q.isActive:
            q.stop()
            print("Consulta {} detenida.".format(q.name))


# ---------------------------------------------------------------- seguimiento
def revisar(consultas):
    for q in consultas:
        if q.exception() is not None:
            raise RuntimeError("La consulta {} fallo: {}".format(q.name, q.exception()))
        if not q.isActive:
            raise RuntimeError("La consulta {} se detuvo inesperadamente.".format(q.name))


def esperar_listas(consultas, limite=120):
    """Espera a que cada consulta haya fijado sus offsets iniciales y este esperando datos."""
    inicio = time.time()
    while time.time() - inicio < limite:
        revisar(consultas)
        if all(q.status["message"].startswith("Waiting") for q in consultas):
            time.sleep(3)
            print("Consultas listas en {:.0f} s: {}".format(time.time() - inicio, ", ".join(q.name for q in consultas)))
            return
        time.sleep(1)
    raise RuntimeError("Las consultas no quedaron listas en {} s. Informar a la docente.".format(limite))


def lanzar_productor(modo, etiqueta):
    resumen = os.path.join(T11, etiqueta + "_productor.json")
    registro = os.path.join(T11, etiqueta + "_productor.log")
    if os.path.exists(resumen):
        os.remove(resumen)
    proceso = subprocess.Popen(
        [sys.executable, PRODUCTOR, "--modo", modo, "--semilla", str(SEMILLA),
         "--bootstrap", BS, "--topic", TOPIC, "--salida", resumen],
        stdout=open(registro, "w"), stderr=subprocess.STDOUT)
    print("Productor lanzado como proceso aparte del sistema operativo: PID {}, modo {}.".format(proceso.pid, modo))
    return proceso, resumen, registro


def por_lote(q, campo):
    """Valor de un campo por numero de lote, a partir de los informes de progreso de la consulta."""
    valores = {}
    for p in q.recentProgress:
        if campo == "descartadas":
            v = sum(op.get("numRowsDroppedByWatermark", 0) for op in p.get("stateOperators", []))
        else:
            v = p.get(campo, 0)
        valores[p["batchId"]] = max(valores.get(p["batchId"], 0), v)
    return valores


def filas_leidas(q):
    return sum(por_lote(q, "numInputRows").values())


def ultimo_lote(q):
    return q.lastProgress["batchId"] if q.lastProgress else "-"


def seguir(consultas, productor):
    """Muestra el avance mientras el productor trabaja y espera a que las consultas lean todo."""
    proceso, ruta_resumen, ruta_registro = productor
    inicio = time.time()
    aviso = -10
    while proceso.poll() is None:
        revisar(consultas)
        t = time.time() - inicio
        if t - aviso >= 10:
            print("  t = {:>3.0f} s | productor enviando | ".format(t) + " | ".join(
                "{}: lote {}, {} filas leidas".format(q.name, ultimo_lote(q), filas_leidas(q)) for q in consultas))
            aviso = t
        time.sleep(1)
    print("  t = {:>3.0f} s | productor terminado (codigo de salida {})".format(time.time() - inicio, proceso.returncode))
    if not os.path.exists(ruta_resumen):
        with open(ruta_registro) as f:
            print(f.read()[-2000:])
        raise RuntimeError("El productor no dejo resumen. Informar a la docente.")
    with open(ruta_resumen) as f:
        r = json.load(f)
    if "error" in r:
        raise RuntimeError("El productor fallo: {}. Informar a la docente.".format(r["error"]))
    esperado = r["total_enviado"] - r.get("errores", 0)
    limite = time.time() + 60
    while time.time() < limite and any(filas_leidas(q) < esperado for q in consultas):
        revisar(consultas)
        time.sleep(1)
    time.sleep(15)      # tres triggers mas: el watermark final cierra las ventanas que pueda cerrar
    print("  t = {:>3.0f} s | lectura completa".format(time.time() - inicio))
    return r


# ---------------------------------------------------------------- informes
class Informe:
    """Imprime y a la vez guarda el resumen de una ejecucion."""

    def __init__(self, etiqueta):
        self.etiqueta, self.lineas = etiqueta, []
        print()
        for s in ("=" * 78, linea_encabezado(etiqueta + " | RESUMEN"), "=" * 78):
            self.p(s)

    def p(self, s=""):
        self.lineas.append(s)
        print(s)

    def guardar(self):
        with open(os.path.join(T11, self.etiqueta + "_resumen.txt"), "w") as f:
            f.write("\n".join(self.lineas) + "\n")


def informe_productor(inf, r):
    inf.p("PRODUCTOR ({})".format(r.get("cliente", "-")))
    inf.p("  Semilla: {}   Inicio: {}   Fin: {}".format(r["semilla"], r["inicio"], r["fin"]))
    inf.p("  Lecturas a tiempo: {}   Lecturas de la rafaga: {}   Total enviado: {}   Errores de envio: {}".format(
        r["lecturas_a_tiempo"], r["lecturas_rafaga"], r["total_enviado"], r.get("errores", 0)))
    if r.get("estaciones_rafaga"):
        detalle = ", ".join("{} ({})".format(e, r["lecturas_por_estacion_rafaga"][e]) for e in r["estaciones_rafaga"])
        inf.p("  Estaciones de la rafaga: {} -> {}".format(len(r["estaciones_rafaga"]), detalle))
        inf.p("  Tiempo de evento de la rafaga: tramo {}   (ventana tumbling {})".format(
            r["tramo_rafaga"], r["ventana_tumbling_rafaga"]))
    inf.p()


def informe_lectura(inf, consultas, r):
    esperado = r["total_enviado"] - r.get("errores", 0)
    inf.p("LECTURA DE LAS CONSULTAS")
    for q in consultas:
        n = filas_leidas(q)
        inf.p("  {:<12} filas leidas: {:>5}   {}".format(
            q.name, n, "coincide con lo enviado" if n == esperado else "NO COINCIDE con lo enviado: informar a la docente"))
    inf.p()


def advertir_descartes(inf, q, se_esperan):
    lotes = [b for b, n in por_lote(q, "descartadas").items() if n > 0]
    if se_esperan and not lotes:
        inf.p("ATENCION: {} no registro filas descartadas por el watermark. Informar a la docente.".format(q.name))
    elif se_esperan and len(lotes) > 1:
        inf.p("ATENCION: los datos tardios de {} se procesaron en mas de un lote. Informar a la docente.".format(q.name))
    elif not se_esperan and lotes:
        inf.p("ATENCION: {} registro filas descartadas por el watermark. Informar a la docente.".format(q.name))


def ventanas_en_memoria(tabla):
    filas = spark.sql("SELECT ventana.start AS inicio, ventana.end AS fin, COUNT(*) AS estaciones, "
                      "SUM(lecturas) AS lecturas FROM {} GROUP BY ventana ORDER BY inicio".format(tabla)).collect()
    return {(f["inicio"], f["fin"]): (f["estaciones"], f["lecturas"]) for f in filas}


def informe_e2(consultas, r):
    inf = Informe("E2")
    informe_productor(inf, r)
    informe_lectura(inf, consultas, r)
    completo, anexar = ventanas_en_memoria("e2_completo"), ventanas_en_memoria("e2_anexar")
    rafaga = datetime.datetime.fromtimestamp(r["inicio_tramo_rafaga"]).strftime("%H:%M:%S")
    inf.p("TABLAS EN MEMORIA, POR VENTANA            e2_completo          e2_anexar")
    inf.p("  Ventana (inicio - fin)               estaciones lecturas  estaciones lecturas")
    for clave in sorted(set(completo) | set(anexar)):
        ini, fin = clave
        c = completo.get(clave)
        a = anexar.get(clave)
        marca = "  (*)" if ini.strftime("%H:%M:%S") == rafaga else ""
        inf.p("  {} - {}{:<14} {:>8} {:>8}  {:>10} {:>8}".format(
            ini.strftime("%H:%M:%S"), fin.strftime("%H:%M:%S"), marca,
            c[0] if c else "-", c[1] if c else "-", a[0] if a else "-", a[1] if a else "-"))
    inf.p("  (*) ventana que contiene el tiempo de evento de la rafaga.  - : la ventana no aparece en esa tabla (no es un error)")
    inf.p()
    advertir_descartes(inf, consultas[0], se_esperan=False)
    advertir_descartes(inf, consultas[1], se_esperan=True)
    inf.guardar()


def informe_e3(etiqueta, q, r):
    inf = Informe(etiqueta)
    informe_productor(inf, r)
    informe_lectura(inf, [q], r)
    lotes = sorted(por_lote(q, "numInputRows"))
    inf.p("CONSULTA e3_ventanas")
    inf.p("  Id:    {}".format(q.id))
    inf.p("  RunId: {}".format(q.runId))
    inf.p("  Lotes ejecutados en esta corrida: del {} al {}".format(lotes[0], lotes[-1]))
    inf.p()
    offsets = lotes_en(CHECKPOINT_E3 + "/offsets")
    commits = lotes_en(CHECKPOINT_E3 + "/commits")
    inf.p("CHECKPOINT ({})".format(CHECKPOINT_E3))
    inf.p("  Contenido: {}".format(", ".join(listar(CHECKPOINT_E3))))
    inf.p("  offsets/: {} archivos (lotes {} a {})".format(len(offsets), offsets[0], offsets[-1]))
    inf.p("  commits/: {} archivos (lotes {} a {})".format(len(commits), commits[0], commits[-1]))
    inf.p()
    archivos = [n for n in listar(SALIDA_E3) if n.endswith(".parquet")]
    salida = spark.read.parquet(SALIDA_E3)
    inf.p("SALIDA PARQUET ({})".format(SALIDA_E3))
    inf.p("  Archivos .parquet: {}   Filas: {}   Ventanas distintas: {}".format(
        len(archivos), salida.count(), salida.select("inicio").distinct().count()))
    inf.p()
    if etiqueta == "E3a":
        advertir_descartes(inf, q, se_esperan=True)
    inf.guardar()
    with open(os.path.join(T11, etiqueta + "_consulta.json"), "w") as f:
        json.dump({"id": q.id, "runId": q.runId, "ultimo_commit": commits[-1]}, f)


print("Funciones cargadas.")

Funciones cargadas.


## Preparación (P0): servicios de Hadoop y Spark

La celda revisa, con `jps`, que estén activos los procesos que necesita el taller en los tres nodos. Si alguno aparece como `NO ACTIVO`, iniciarlo y ejecutar la celda de nuevo.

In [3]:
encabezado("P0")
procesos = {}
for nodo in ("local", "nodo2", "nodo3"):
    cmd = ["jps"] if nodo == "local" else ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=5", nodo,
                                           "/opt/hadoop/jdk/bin/jps"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    procesos[nodo] = set(l.split()[-1] for l in r.stdout.splitlines() if len(l.split()) == 2)

revision = [
    ("local", "NameNode",        "HDFS: NameNode (maestro)"),
    ("local", "ResourceManager", "YARN: ResourceManager (maestro)"),
    ("local", "HistoryServer",   "History Server de Spark (maestro)"),
    ("nodo2", "DataNode",        "HDFS: DataNode (nodo2)"),
    ("nodo2", "NodeManager",     "YARN: NodeManager (nodo2)"),
    ("nodo3", "DataNode",        "HDFS: DataNode (nodo3)"),
    ("nodo3", "NodeManager",     "YARN: NodeManager (nodo3)"),
]
faltan = 0
for nodo, proceso, nombre in revision:
    activo = proceso in procesos[nodo]
    faltan += 0 if activo else 1
    print("  {:<36} {}".format(nombre, "activo" if activo else "NO ACTIVO"))
print()
print("Servicios listos." if faltan == 0 else
      "ATENCION: iniciar los servicios que faltan, como en los Talleres 7 y 8, y ejecutar de nuevo esta celda.")

P0 | 2026-10-30 16:55:30 | sin sesion de Spark | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
  HDFS: NameNode (maestro)             activo
  YARN: ResourceManager (maestro)      activo
  History Server de Spark (maestro)    activo
  HDFS: DataNode (nodo2)               activo
  YARN: NodeManager (nodo2)            activo
  HDFS: DataNode (nodo3)               activo
  YARN: NodeManager (nodo3)            activo

Servicios listos.


## Preparación (P1): clúster de Kafka

La celda siguiente escribe el script `~/kafka_cluster.sh`, el mismo del Taller 10 con un ajuste: ya no sugiere detener HDFS y YARN, que en este taller sí se usan. La siguiente lo ejecuta con la opción `iniciar`:

- Si los tres brokers ya están activos y responden, **no los reinicia**: solo verifica.
- Si no lo están, los arranca. Si el almacenamiento no está formateado (por ejemplo, porque el Taller 10 no se ejecutó en este equipo), formatea los tres nodos con un `ClusterId` nuevo.

Debe terminar con la línea **`RESULTADO: CLUSTER LISTO`**. Si termina con `CLUSTER NO LISTO`, esperar 20 segundos y ejecutarla una segunda vez. Si el resultado se repite, informar a la docente.

In [4]:
%%writefile /home/hadoop/kafka_cluster.sh
#!/bin/bash
# =====================================================================
# kafka_cluster.sh - Cluster Kafka 3.8.1 en modo KRaft (tres nodos)
# Big Data 2026-II - Universidad Externado de Colombia
#
# Se ejecuta SIEMPRE desde el nodo maestro, con el usuario hadoop:
#
#   bash ~/kafka_cluster.sh iniciar      Formatea solo si hace falta, arranca
#                                        los tres brokers y verifica el cluster.
#   bash ~/kafka_cluster.sh verificar    Solo informa el estado.
#   bash ~/kafka_cluster.sh detener      Detiene los tres brokers.
#   bash ~/kafka_cluster.sh reformatear  BORRA todos los topics y grupos,
#                                        formatea de nuevo y arranca.
#
# Heap de los brokers: por defecto, kafka-server-start.sh usa -Xmx1G -Xms1G.
# Para cambiarlo:
#   KAFKA_HEAP_OPTS="-Xmx512M -Xms512M" bash ~/kafka_cluster.sh iniciar
#
# Por que "iniciar" formatea cuando hace falta: en KRaft, cada nodo guarda en
# meta.properties el cluster.id con el que se formateo. Los tres nodos deben
# compartir el mismo cluster.id y cada uno debe estar formateado con su propio
# archivo de configuracion (node.id 1, 2 y 3). Si falta meta.properties en
# algun nodo, o si los nodos no coinciden, el script formatea los tres con un
# cluster.id nuevo. Si todo coincide, conserva los datos.
#
# Por que los tres brokers arrancan casi a la vez: cada nodo es broker y
# controller. Un broker que no logra registrarse ante el quorum de controllers
# en initial.broker.registration.timeout.ms (60 s por defecto) termina su
# proceso. Con tres controllers, el quorum exige al menos dos activos.
# =====================================================================

export JAVA_HOME=/opt/hadoop/jdk
export PATH="$JAVA_HOME/bin:$PATH"

KAFKA=/opt/kafka
BOOTSTRAP="192.168.56.101:9092,192.168.56.102:9092,192.168.56.103:9092"
NOMBRES=("maestro (192.168.56.101)" "nodo2   (192.168.56.102)" "nodo3   (192.168.56.103)")
HOSTS=(local nodo2 nodo3)
CONFS=(maestro-server.properties worker1-server.properties worker2-server.properties)
ESPERA_MAX=120
SSH_OPTS="-o BatchMode=yes -o ConnectTimeout=5"

# ---------------------------------------------------------------------
# Utilidades
# ---------------------------------------------------------------------

linea() { echo "--------------------------------------------------------------------"; }

# en_nodo <indice> <comando>: ejecuta el comando en el nodo indicado.
en_nodo() {
    local i=$1
    shift
    if [ "$i" -eq 0 ]; then
        bash -c "$*"
    else
        local heap=""
        [ -n "${KAFKA_HEAP_OPTS:-}" ] && heap="export KAFKA_HEAP_OPTS='${KAFKA_HEAP_OPTS}';"
        # shellcheck disable=SC2086
        ssh $SSH_OPTS "hadoop@${HOSTS[$i]}" \
            "export JAVA_HOME=$JAVA_HOME; export PATH=\$JAVA_HOME/bin:\$PATH; $heap $*"
    fi
}

conf_de() { echo "$KAFKA/config/kraft/${CONFS[$1]}"; }

# Valor de una propiedad en el archivo de configuracion del nodo.
propiedad() {
    en_nodo "$1" "grep -E '^$2=' $(conf_de "$1") 2>/dev/null | head -1 | cut -d= -f2-"
}

# Directorios de datos del nodo: log.dirs y, si existe, metadata.log.dir.
dirs_datos() {
    local d m
    d=$(propiedad "$1" 'log\.dirs' | tr ',' ' ')
    m=$(propiedad "$1" 'metadata\.log\.dir')
    echo "$d $m" | xargs
}

# Propiedad de meta.properties (cluster.id o node.id) en el primer log.dir.
meta_de() {
    local d
    d=$(dirs_datos "$1" | awk '{print $1}')
    [ -z "$d" ] && return
    en_nodo "$1" "grep -s '^$2=' $d/meta.properties | cut -d= -f2"
}

pid_kafka() { en_nodo "$1" "pgrep -f 'kafka[.]Kafka' | head -1"; }

# Brokers que responden: se cuentan las lineas "(id: N" sin error de conexion.
# Un broker recien detenido puede seguir apareciendo en los metadatos unos
# segundos, pero con ERROR (por ejemplo, DisconnectException): no se cuenta.
brokers_registrados() {
    timeout 25 "$KAFKA/bin/kafka-broker-api-versions.sh" --bootstrap-server "$BOOTSTRAP" 2>/dev/null \
        | grep '(id: ' | grep -vc 'ERROR'
}

# Devuelve 0 si el cluster ya esta sano: tres procesos, almacenamiento
# coherente y tres brokers que responden.
cluster_sano() {
    local i
    for i in 0 1 2; do
        [ -z "$(pid_kafka "$i")" ] && return 1
    done
    almacenamiento_coherente >/dev/null || return 1
    [ "$(brokers_registrados)" -eq 3 ]
}

# ---------------------------------------------------------------------
# Comprobaciones previas
# ---------------------------------------------------------------------

comprobar_entorno() {
    local ok=0 i
    if ! hostname -I 2>/dev/null | grep -qw '192.168.56.101'; then
        echo "ERROR: este script debe ejecutarse en el nodo maestro (192.168.56.101)."
        exit 1
    fi
    for i in 1 2; do
        if ! en_nodo "$i" true 2>/dev/null; then
            echo "ERROR: no hay conexion SSH con ${HOSTS[$i]}. Verificar que la maquina virtual este encendida."
            ok=1
        fi
    done
    [ $ok -ne 0 ] && exit 1
    for i in 0 1 2; do
        if ! en_nodo "$i" "test -f $(conf_de "$i")"; then
            echo "ERROR: falta $(conf_de "$i") en ${NOMBRES[$i]}."
            ok=1
        fi
        if ! en_nodo "$i" "test -x $JAVA_HOME/bin/java"; then
            echo "ERROR: no se encuentra Java en $JAVA_HOME en ${NOMBRES[$i]}."
            ok=1
        fi
        if [ -z "$(dirs_datos "$i")" ]; then
            echo "ERROR: $(conf_de "$i") no define log.dirs en ${NOMBRES[$i]}."
            ok=1
        fi
    done
    [ $ok -ne 0 ] && exit 1
}

# ---------------------------------------------------------------------
# Detencion
# ---------------------------------------------------------------------

detener_brokers() {
    local i vivos
    for i in 0 1 2; do
        en_nodo "$i" "$KAFKA/bin/kafka-server-stop.sh >/dev/null 2>&1"
    done
    for _ in $(seq 1 15); do
        vivos=0
        for i in 0 1 2; do
            [ -n "$(pid_kafka "$i")" ] && vivos=$((vivos + 1))
        done
        [ $vivos -eq 0 ] && return 0
        sleep 2
    done
    echo "  Algunos brokers no terminaron en 30 s (puede ocurrir al detener los tres nodos a la vez,"
    echo "  porque el ultimo pierde el quorum). Se detienen de forma forzada; los mensajes ya escritos se conservan."
    for i in 0 1 2; do
        en_nodo "$i" "pkill -9 -f 'kafka[.]Kafka'"
    done
    sleep 3
}

# ---------------------------------------------------------------------
# Formato del almacenamiento
# ---------------------------------------------------------------------

# Devuelve 0 si los tres nodos estan formateados con el mismo cluster.id
# y con el node.id de su propio archivo de configuracion.
almacenamiento_coherente() {
    local i cid nid nconf primero=""
    for i in 0 1 2; do
        cid=$(meta_de "$i" cluster.id)
        nid=$(meta_de "$i" node.id)
        nconf=$(propiedad "$i" 'node\.id')
        printf "  %s  cluster.id=%-24s node.id=%s (configuracion: %s)\n" \
            "${NOMBRES[$i]}" "${cid:-SIN_FORMATO}" "${nid:--}" "$nconf"
        [ -z "$cid" ] && return 1
        [ "$nid" != "$nconf" ] && return 1
        [ -z "$primero" ] && primero=$cid
        [ "$cid" != "$primero" ] && return 1
    done
    return 0
}

formatear() {
    local uuid i dirs d
    uuid=$("$KAFKA/bin/kafka-storage.sh" random-uuid 2>/dev/null | tail -1)
    if [ -z "$uuid" ]; then
        echo "ERROR: no fue posible generar el cluster.id (kafka-storage.sh random-uuid)."
        exit 1
    fi
    echo "  cluster.id nuevo: $uuid"
    if ls /home/hadoop/t10/*.hecho >/dev/null 2>&1; then
        rm -f /home/hadoop/t10/*.hecho
        echo "  Se eliminaron las marcas del Taller 10 (~/t10/*.hecho): el notebook puede ejecutarse de nuevo desde E1."
    fi
    for i in 0 1 2; do
        dirs=$(dirs_datos "$i")
        for d in $dirs; do
            case "$d" in
                *kafka*) ;;
                *) echo "ERROR: se rehusa borrar '$d' en ${NOMBRES[$i]} (la ruta no contiene 'kafka')."; exit 1 ;;
            esac
        done
        echo "  Formateando ${NOMBRES[$i]} con $(basename "$(conf_de "$i")")..."
        if ! en_nodo "$i" "set -o pipefail; for d in $dirs; do mkdir -p \$d && find \$d -mindepth 1 -delete; done && \
                $KAFKA/bin/kafka-storage.sh format -t $uuid -c $(conf_de "$i") 2>&1 | sed 's/^/    /'"; then
            echo "ERROR: fallo el formato en ${NOMBRES[$i]}."
            exit 1
        fi
    done
}

# ---------------------------------------------------------------------
# Arranque y espera
# ---------------------------------------------------------------------

arrancar_y_esperar() {
    local i t n pid ok_seguidas=0
    echo "Arrancando los tres brokers (nodo2, nodo3 y maestro, sin pausas)..."
    for i in 1 2 0; do
        en_nodo "$i" "cd $KAFKA && bin/kafka-server-start.sh -daemon $(conf_de "$i")"
    done
    echo
    echo "Esperando a que el cluster se forme (maximo $ESPERA_MAX s)..."
    t=0
    while [ $t -lt $ESPERA_MAX ]; do
        sleep 5
        t=$((t + 5))
        for i in 0 1 2; do
            pid=$(pid_kafka "$i")
            if [ -z "$pid" ] && [ $t -ge 10 ]; then
                echo
                echo "ERROR: el broker de ${NOMBRES[$i]} se detuvo. Ultimos errores de su registro:"
                en_nodo "$i" "grep -E 'ERROR|FATAL|Exception' $KAFKA/logs/kafkaServer.out 2>/dev/null | tail -n 8" | sed 's/^/    /'
                return 1
            fi
        done
        n=$(brokers_registrados)
        echo "  ${t} s: brokers que responden ${n:-0} de 3"
        if [ "${n:-0}" -eq 3 ]; then
            ok_seguidas=$((ok_seguidas + 1))
            [ $ok_seguidas -ge 2 ] && return 0
        else
            ok_seguidas=0
        fi
    done
    echo "ERROR: el cluster no se formo en $ESPERA_MAX s."
    return 1
}

# ---------------------------------------------------------------------
# Verificacion
# ---------------------------------------------------------------------

verificar() {
    local i pid vivos=0 q lider votantes n cids cid servicios resultado=0
    linea
    echo "VERIFICACION DEL CLUSTER KAFKA    $(date '+%Y-%m-%d %H:%M:%S')"
    linea
    echo "Procesos Kafka:"
    for i in 0 1 2; do
        pid=$(pid_kafka "$i")
        if [ -n "$pid" ]; then
            echo "  ${NOMBRES[$i]}  activo (PID $pid)"
            vivos=$((vivos + 1))
        else
            echo "  ${NOMBRES[$i]}  DETENIDO"
        fi
    done

    echo
    echo "cluster.id en meta.properties:"
    cids=""
    for i in 0 1 2; do
        cid=$(meta_de "$i" cluster.id)
        echo "  ${NOMBRES[$i]}  ${cid:-SIN_FORMATO}"
        cids="$cids $cid"
    done
    if [ "$(echo "$cids" | xargs -n1 | sort -u | wc -l)" -ne 1 ]; then
        echo "  ATENCION: los tres nodos no comparten el mismo cluster.id."
        resultado=1
    fi

    echo
    echo "Quorum KRaft (kafka-metadata-quorum.sh describe --status):"
    q=$(timeout 25 "$KAFKA/bin/kafka-metadata-quorum.sh" --bootstrap-server "$BOOTSTRAP" describe --status 2>&1)
    echo "$q" | sed 's/^/  /'
    lider=$(echo "$q" | awk '/^LeaderId:/{print $2}')
    votantes=$(echo "$q" | awk '/^CurrentVoters:/{print $2}' | tr -d '[]' | tr ',' '\n' | sort -n | xargs)

    echo
    echo "Brokers registrados:"
    timeout 25 "$KAFKA/bin/kafka-broker-api-versions.sh" --bootstrap-server "$BOOTSTRAP" 2>/dev/null \
        | grep '(id: ' | sed 's/ -> ($//; s/ -> ERROR.*/  NO RESPONDE/; s/^/  /'
    n=$(brokers_registrados)

    echo
    echo "Memoria del maestro:"
    free -h | head -2 | sed 's/^/  /'
    servicios=$(jps 2>/dev/null | awk '$2 ~ /^(NameNode|SecondaryNameNode|ResourceManager|HistoryServer|RunJar)$/ {print $2}' | xargs)
    pgrep -f 'airflow' >/dev/null 2>&1 && servicios="$servicios airflow"
    if [ -n "$(echo "$servicios" | xargs)" ]; then
        echo "  Otros servicios activos en el maestro: $(echo "$servicios" | xargs)"
    fi

    echo
    linea
    if [ $vivos -eq 3 ] && [ "${n:-0}" -eq 3 ] && [ -n "$lider" ] && [ "$lider" != "-1" ] \
        && [ "$votantes" = "1 2 3" ] && [ $resultado -eq 0 ]; then
        echo "RESULTADO: CLUSTER LISTO. Tres brokers activos; lider del quorum: nodo $lider."
        linea
        return 0
    else
        echo "RESULTADO: CLUSTER NO LISTO. Revisar los mensajes anteriores."
        echo "Esperar 20 segundos y ejecutar: bash ~/kafka_cluster.sh verificar"
        echo "Si sigue sin estar listo: bash ~/kafka_cluster.sh iniciar"
        linea
        return 1
    fi
}

# ---------------------------------------------------------------------
# Programa principal
# ---------------------------------------------------------------------

case "${1:-}" in
    iniciar)
        comprobar_entorno
        if cluster_sano; then
            echo "Los tres brokers ya estan activos, responden y comparten el mismo cluster.id."
            echo "No se reinician: se conservan los topics, los grupos y el ClusterId."
            echo
            verificar
            exit $?
        fi
        echo "Deteniendo procesos Kafka previos, si los hay..."
        detener_brokers
        echo
        echo "Estado del almacenamiento:"
        if almacenamiento_coherente; then
            echo "  Los tres nodos estan formateados con el mismo cluster.id. Se conservan los topics."
        else
            echo "  Falta el formato o los nodos no coinciden. Se formatean los tres con un cluster.id nuevo."
            formatear
        fi
        echo
        arrancar_y_esperar
        echo
        verificar
        ;;
    verificar)
        comprobar_entorno
        verificar
        ;;
    detener)
        comprobar_entorno
        echo "Deteniendo los tres brokers..."
        detener_brokers
        for i in 0 1 2; do
            if [ -n "$(pid_kafka "$i")" ]; then echo "  ${NOMBRES[$i]}  sigue activo"; else echo "  ${NOMBRES[$i]}  detenido"; fi
        done
        ;;
    reformatear)
        comprobar_entorno
        echo "ATENCION: se borraran todos los topics, mensajes y grupos de consumidores."
        echo "Para cancelar, presionar Ctrl+C en los proximos 10 segundos."
        sleep 10
        detener_brokers
        formatear
        echo
        arrancar_y_esperar
        echo
        verificar
        ;;
    *)
        echo "Uso: bash ~/kafka_cluster.sh {iniciar|verificar|detener|reformatear}"
        exit 2
        ;;
esac

Overwriting /home/hadoop/kafka_cluster.sh


In [5]:
!bash /home/hadoop/kafka_cluster.sh iniciar

Deteniendo procesos Kafka previos, si los hay...

Estado del almacenamiento:
  maestro (192.168.56.101)  cluster.id=Rxkee_6WRYaGnf3eVYjxNw   node.id=1 (configuracion: 1)
  nodo2   (192.168.56.102)  cluster.id=Rxkee_6WRYaGnf3eVYjxNw   node.id=2 (configuracion: 2)
  nodo3   (192.168.56.103)  cluster.id=Rxkee_6WRYaGnf3eVYjxNw   node.id=3 (configuracion: 3)
  Los tres nodos estan formateados con el mismo cluster.id. Se conservan los topics.

Arrancando los tres brokers (nodo2, nodo3 y maestro, sin pausas)...

Esperando a que el cluster se forme (maximo 120 s)...
  5 s: brokers que responden 3 de 3
  10 s: brokers que responden 3 de 3

--------------------------------------------------------------------
VERIFICACION DEL CLUSTER KAFKA    2026-10-30 16:56:29
--------------------------------------------------------------------
Procesos Kafka:
  maestro (192.168.56.101)  activo (PID 8295)
  nodo2   (192.168.56.102)  activo (PID 3089)
  nodo3   (192.168.56.103)  activo (PID 3045)

cluster.id en 

## Preparación (P2): conector de Kafka para Spark y cliente de Kafka para Python

**El conector.** Spark no trae incorporada la lectura desde Kafka: la aporta un conector, que debe coincidir con la versión de Spark (3.5.7) y con su versión de Scala (2.12). Son cuatro archivos `.jar`: el conector, un módulo de credenciales que el conector exige, el cliente de Kafka para Java en la versión con la que se compiló el conector (3.4.1) y una biblioteca de agrupación de conexiones. La celda los descarga de Maven Central y verifica cada uno con su huella SHA-1 publicada. Si ya están descargados y verificados, no los descarga de nuevo.

**El cliente de Python.** El productor usa `kafka-python`, fijado en la versión 2.2.21. Si no queda instalado, el productor recurre a un plan B: envía los mensajes a través de `kafka-console-producer.sh`. El resumen de cada ejecución indica qué cliente se usó.

In [6]:
encabezado("P2")
import hashlib, urllib.request

REPOSITORIOS = [
    "https://repo1.maven.org/maven2",
    "https://repo.maven.apache.org/maven2",
    "https://maven-central.storage-download.googleapis.com/maven2",
]


def sha1(ruta):
    h = hashlib.sha1()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def descargar(grupo, artefacto, version):
    destino = "{}/{}-{}.jar".format(JARS_DIR, artefacto, version)
    marca = destino + ".sha1"
    if os.path.exists(destino) and os.path.exists(marca):
        with open(marca) as f:
            if f.read().strip() == sha1(destino):
                return "ya estaba (SHA-1 verificado)"
    ultimo = "el SHA-1 no coincide"
    for repo in REPOSITORIOS:
        url = "{}/{}/{}/{}/{}-{}.jar".format(repo, grupo, artefacto, version, artefacto, version)
        try:
            esperado = urllib.request.urlopen(url + ".sha1", timeout=30).read().decode().split()[0].strip()
            urllib.request.urlretrieve(url, destino)
            if sha1(destino) == esperado:
                with open(marca, "w") as f:
                    f.write(esperado)
                return "descargado de {} (SHA-1 verificado)".format(repo.split("/")[2])
            os.remove(destino)
        except Exception as error:
            ultimo = error
    return "FALLO: {}".format(ultimo)


print("Conector de Kafka para Spark (carpeta {}):".format(JARS_DIR))
fallos = 0
for grupo, artefacto, version in ARTEFACTOS:
    estado = descargar(grupo, artefacto, version)
    fallos += estado.startswith("FALLO")
    print("  {:<48} {}".format("{}-{}.jar".format(artefacto, version), estado))

print()
print("Cliente de Kafka para Python (kafka-python 2.2.21):")
comprobar = [sys.executable, "-c", "import kafka; print(kafka.__version__)"]
r = subprocess.run(comprobar, capture_output=True, text=True)
if r.stdout.strip() != "2.2.21":
    en_entorno_virtual = sys.prefix != sys.base_prefix
    instalar = [sys.executable, "-m", "pip", "install", "--quiet", "kafka-python==2.2.21"]
    if not en_entorno_virtual:
        instalar.append("--user")
    subprocess.run(instalar, capture_output=True, text=True)
    r = subprocess.run(comprobar, capture_output=True, text=True)
if r.stdout.strip() == "2.2.21":
    print("  kafka-python", r.stdout.strip(), "disponible para", sys.executable)
else:
    print("  kafka-python no quedo instalado. El productor usara el plan B (kafka-console-producer.sh).")

print()
print("Dependencias listas." if fallos == 0 else "ATENCION: faltan jars del conector. Informar a la docente antes de continuar.")

P2 | 2026-10-30 16:57:25 | sin sesion de Spark | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
Conector de Kafka para Spark (carpeta /home/hadoop/t11/jars):
  spark-sql-kafka-0-10_2.12-3.5.7.jar              descargado de repo1.maven.org (SHA-1 verificado)
  spark-token-provider-kafka-0-10_2.12-3.5.7.jar   descargado de repo1.maven.org (SHA-1 verificado)
  kafka-clients-3.4.1.jar                          descargado de repo1.maven.org (SHA-1 verificado)
  commons-pool2-2.11.1.jar                         descargado de repo1.maven.org (SHA-1 verificado)

Cliente de Kafka para Python (kafka-python 2.2.21):
  kafka-python 2.2.21 disponible para /usr/bin/python3

Dependencias listas.


**P2-alternativa (solo si P2 muestra FALLO y la docente lo indica).** Descargar `t11_offline.zip` desde el enlace de OneDrive publicado en Teams, con el navegador de esta máquina virtual, y ejecutar la celda siguiente. Después, ejecutar P2 de nuevo.

In [7]:
encabezado("P2-alternativa")
import glob, zipfile
candidatos = []
for carpeta in ("~/Descargas", "~/Downloads"):
    candidatos += glob.glob(os.path.expanduser(carpeta) + "/t11_offline*.zip")
if not candidatos:
    print("ATENCION: no se encontro t11_offline.zip en Descargas. Descargarlo desde OneDrive y ejecutar de nuevo esta celda.")
else:
    paquete = max(candidatos, key=os.path.getmtime)
    print("Paquete:", paquete)
    with zipfile.ZipFile(paquete) as z:
        for miembro in z.namelist():
            nombre = os.path.basename(miembro)
            if nombre.endswith((".jar", ".jar.sha1")):
                with open(os.path.join(JARS_DIR, nombre), "wb") as f:
                    f.write(z.read(miembro))
                print("  copiado a", JARS_DIR + ":", nombre)
            elif nombre.endswith(".whl"):
                ruta_rueda = os.path.join(T11, nombre)
                with open(ruta_rueda, "wb") as f:
                    f.write(z.read(miembro))
                instalar = [sys.executable, "-m", "pip", "install", "--quiet", "--no-index", ruta_rueda]
                if sys.prefix == sys.base_prefix:
                    instalar.append("--user")
                r = subprocess.run(instalar, capture_output=True, text=True)
                print("  kafka-python:", "instalado" if r.returncode == 0 else "no se instalo (el productor usara el plan B)")
    print()
    print("Ejecutar de nuevo la celda P2: debe indicar 'ya estaba (SHA-1 verificado)' en los cuatro jars.")

P2-alternativa | 2026-10-30 16:57:34 | sin sesion de Spark | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
ATENCION: no se encontro t11_offline.zip en Descargas. Descargarlo desde OneDrive y ejecutar de nuevo esta celda.


## Preparación (P3): el productor y la semilla

La primera celda escribe el productor en `~/t11/productor.py`. No es necesario ejecutarlo a mano: lo lanzan las celdas E2, E3a y E3b. Conviene leer sus comentarios iniciales, que describen qué envía.

La segunda celda fija la **semilla** de este clúster, un número al azar que se guarda en `~/t11/semilla.txt`. La semilla decide qué estaciones participan en la ráfaga de lecturas tardías y cuántas lecturas envía cada una, así que la ráfaga es distinta en cada estudiante pero idéntica en todas las ejecuciones del mismo estudiante.

In [8]:
%%writefile /home/hadoop/t11/productor.py
#!/usr/bin/env python3
# =====================================================================
# productor.py - Lecturas simuladas de PM2.5 para el Taller 11
# Big Data 2026-II - Universidad Externado de Colombia
#
# Lo lanza el notebook como un proceso aparte; no es necesario ejecutarlo a mano.
#
#   python3 productor.py --modo completo --semilla 1234 --salida resumen.json
#   python3 productor.py --modo corto    --semilla 1234 --salida resumen.json
#
# Modo completo (unos 80 s):
#   Fase A, 30 s: lecturas a tiempo de las ocho estaciones.
#   Rafaga tardia: algunas estaciones envian lecturas cuyo tiempo de evento
#                  quedo unos dos minutos atras. Que estaciones y cuantas
#                  lecturas depende solo de la semilla, asi que se repite
#                  igual en todas las ejecuciones del mismo estudiante.
#   Fase B, 45 s: lecturas a tiempo.
# Modo corto (20 s): solo lecturas a tiempo, sin rafaga.
#
# Cada mensaje tiene como clave el codigo de la estacion y como valor un JSON:
#   {"estacion": "EST-03", "ts_evento_ms": 1793000000123, "pm25": 23.4}
# ts_evento_ms es el TIEMPO DE EVENTO: el instante en que el sensor tomo la
# lectura, en milisegundos desde 1970. Kafka anade por su cuenta otra marca de
# tiempo, la del momento en que recibe el mensaje.
#
# Los datos son sinteticos: no corresponden a ninguna red de monitoreo real.
# =====================================================================

import argparse
import datetime
import json
import os
import random
import subprocess
import sys
import time

KB = "/opt/kafka/bin"
ESTACIONES = ["EST-%02d" % i for i in range(1, 9)]
PM25_BASE = {e: 12 + 4 * i for i, e in enumerate(ESTACIONES)}


def hora(segundos):
    return datetime.datetime.fromtimestamp(segundos).strftime("%H:%M:%S")


class Envio:
    """Envia mensajes con kafka-python; si no esta instalado, con el productor de consola de Kafka."""

    def __init__(self, bootstrap, topic):
        self.topic = topic
        self.errores = 0
        self.pendientes = []
        try:
            import kafka
            from kafka import KafkaProducer
        except ImportError:
            kafka = None
        if kafka is not None:
            self.productor = KafkaProducer(
                bootstrap_servers=bootstrap.split(","),
                client_id="productor-t11",
                acks="all",
                linger_ms=5,
                key_serializer=lambda k: k.encode("utf-8"),
                value_serializer=lambda v: json.dumps(v).encode("utf-8"),
            )
            self.consola = None
            self.cliente = "kafka-python " + kafka.__version__
        else:
            # Plan B: el productor de consola lee lineas "clave<TAB>valor" de su entrada estandar.
            self.productor = None
            self.consola = subprocess.Popen(
                [KB + "/kafka-console-producer.sh", "--bootstrap-server", bootstrap, "--topic", topic,
                 "--property", "parse.key=true", "--property", "key.separator=\t"],
                stdin=subprocess.PIPE, text=True)
            self.cliente = "kafka-console-producer.sh (plan B: kafka-python no esta instalado)"

    def enviar(self, clave, valor):
        if self.productor is not None:
            self.pendientes.append(self.productor.send(self.topic, key=clave, value=valor))
        else:
            self.consola.stdin.write(clave + "\t" + json.dumps(valor) + "\n")

    def vaciar(self):
        """Fuerza el envio de lo acumulado y cuenta los mensajes que Kafka no confirmo."""
        if self.productor is not None:
            self.productor.flush()
            for futuro in self.pendientes:
                try:
                    futuro.get(timeout=30)
                except Exception:
                    self.errores += 1
            self.pendientes = []
        else:
            self.consola.stdin.flush()

    def cerrar(self):
        self.vaciar()
        if self.productor is not None:
            self.productor.close()
        else:
            self.consola.stdin.close()
            self.consola.wait(timeout=60)
            if self.consola.returncode != 0:
                self.errores += 1


def fase(envio, segundos, rng, nombre, conteo):
    """Durante 'segundos', cada estacion envia una o dos lecturas por segundo, con tiempo de evento actual."""
    inicio = time.time()
    siguiente = inicio
    ultimo_aviso = inicio
    while time.time() - inicio < segundos:
        ahora_ms = int(time.time() * 1000)
        for est in ESTACIONES:
            for _ in range(rng.choice((1, 1, 2))):
                ts = ahora_ms - rng.randint(0, 1500)          # pequeno desorden, muy por debajo del watermark
                pm25 = round(max(1.0, rng.gauss(PM25_BASE[est], 3)), 1)
                envio.enviar(est, {"estacion": est, "ts_evento_ms": ts, "pm25": pm25})
                conteo["a_tiempo"] += 1
        if time.time() - ultimo_aviso >= 10:
            envio.vaciar()
            print("{} | {} | {} lecturas a tiempo enviadas".format(hora(time.time()), nombre, conteo["a_tiempo"]), flush=True)
            ultimo_aviso = time.time()
        siguiente += 1.0
        time.sleep(max(0.0, siguiente - time.time()))
    envio.vaciar()


def rafaga(envio, semilla, conteo):
    """Lecturas tardias: tiempo de evento unos dos minutos atras, todas dentro de un mismo tramo de 10 s."""
    rng = random.Random(semilla)                     # depende solo de la semilla
    estaciones = sorted(rng.sample(ESTACIONES, rng.randint(3, 6)))
    por_estacion = {e: rng.randint(2, 5) for e in estaciones}
    # Inicio del tramo: multiplo de 30 s (y por lo tanto de 10 s), unos 120 s antes de ahora.
    inicio_tramo = int((time.time() - 120) // 30 * 30)
    for est in estaciones:
        for _ in range(por_estacion[est]):
            ts = inicio_tramo * 1000 + rng.randint(0, 8000)   # siempre antes de inicio_tramo + 10 s
            pm25 = round(max(1.0, rng.gauss(PM25_BASE[est], 3)), 1)
            envio.enviar(est, {"estacion": est, "ts_evento_ms": ts, "pm25": pm25})
            conteo["rafaga"] += 1
    envio.vaciar()
    print("{} | RAFAGA | {} lecturas tardias de {} estaciones, con tiempo de evento entre {} y {}".format(
        hora(time.time()), conteo["rafaga"], len(estaciones), hora(inicio_tramo), hora(inicio_tramo + 9)), flush=True)
    return estaciones, por_estacion, inicio_tramo


def main():
    ap = argparse.ArgumentParser(description="Productor de lecturas simuladas de PM2.5 (Taller 11)")
    ap.add_argument("--modo", choices=["completo", "corto"], required=True)
    ap.add_argument("--semilla", type=int, required=True)
    ap.add_argument("--bootstrap", default="192.168.56.101:9092,192.168.56.102:9092,192.168.56.103:9092")
    ap.add_argument("--topic", default="t11-lecturas")
    ap.add_argument("--salida", required=True, help="archivo JSON donde se escribe el resumen")
    args = ap.parse_args()

    resumen = {"modo": args.modo, "semilla": args.semilla, "pid": os.getpid(), "inicio": hora(time.time())}
    conteo = {"a_tiempo": 0, "rafaga": 0}
    rng = random.Random()                            # lecturas a tiempo: distintas en cada ejecucion
    try:
        envio = Envio(args.bootstrap, args.topic)
        resumen["cliente"] = envio.cliente
        print("{} | Productor iniciado (PID {}) con {}".format(hora(time.time()), os.getpid(), envio.cliente), flush=True)
        if args.modo == "completo":
            fase(envio, 30, rng, "FASE A", conteo)
            estaciones, por_estacion, inicio_tramo = rafaga(envio, args.semilla, conteo)
            resumen.update({
                "estaciones_rafaga": estaciones,
                "lecturas_por_estacion_rafaga": por_estacion,
                "inicio_tramo_rafaga": inicio_tramo,
                "tramo_rafaga": "{} - {}".format(hora(inicio_tramo), hora(inicio_tramo + 10)),
                "ventana_tumbling_rafaga": "{} - {}".format(hora(inicio_tramo), hora(inicio_tramo + 30)),
            })
            fase(envio, 45, rng, "FASE B", conteo)
        else:
            fase(envio, 20, rng, "CORTO", conteo)
        envio.cerrar()
        resumen["errores"] = envio.errores
    except Exception as error:                       # por ejemplo, NoBrokersAvailable
        resumen["error"] = "{}: {}".format(type(error).__name__, error)
        print("ERROR:", resumen["error"], flush=True)
    resumen.update({"fin": hora(time.time()), "lecturas_a_tiempo": conteo["a_tiempo"],
                    "lecturas_rafaga": conteo["rafaga"], "total_enviado": conteo["a_tiempo"] + conteo["rafaga"]})
    with open(args.salida, "w") as f:
        json.dump(resumen, f, indent=2)
    print("{} | Productor terminado: {} lecturas enviadas".format(hora(time.time()), resumen["total_enviado"]), flush=True)
    sys.exit(1 if "error" in resumen or resumen.get("errores") else 0)


if __name__ == "__main__":
    main()

Writing /home/hadoop/t11/productor.py


In [9]:
encabezado("P3")
ruta_semilla = os.path.join(T11, "semilla.txt")
if not os.path.exists(ruta_semilla):
    with open(ruta_semilla, "w") as f:
        f.write(str(random.randint(1000, 9999)))
with open(ruta_semilla) as f:
    SEMILLA = int(f.read().strip())
print("Productor:", PRODUCTOR, "(existe)" if os.path.exists(PRODUCTOR) else "(NO EXISTE: ejecutar la celda anterior)")
print("Semilla de este cluster:", SEMILLA)

P3 | 2026-10-30 16:58:16 | sin sesion de Spark | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
Productor: /home/hadoop/t11/productor.py (existe)
Semilla de este cluster: 5452


---
## Ejecución 1 — La sesión de Spark y el topic

Se lanza la aplicación de Spark sobre YARN con los cuatro jars del conector, y se crea el topic `t11-lecturas` con 3 particiones y factor de replicación 2. Si el topic ya existe, la celda no lo crea de nuevo.

**Por qué `spark.sql.shuffle.partitions` = 4.** Una consulta con estado reparte ese estado (los conteos de cada ventana abierta) en tantas particiones como indique este parámetro, y cada micro-batch lanza una tarea por partición. Con el valor por defecto, 200, cada lote de unos pocos cientos de lecturas lanzaría 200 tareas. Además, el número queda fijado en el checkpoint: una consulta que se reinicia desde su checkpoint no puede cambiarlo.

In [10]:
encabezado("E1")
from pyspark.sql import SparkSession

faltan = [os.path.basename(j) for j in JARS if not os.path.exists(j)]
if faltan:
    print("ATENCION: faltan jars del conector ({}). Ejecutar P2 antes de esta celda.".format(", ".join(faltan)))
else:
    spark = (SparkSession.builder
             .appName("Taller11 - Structured Streaming")
             .master("yarn")
             .config("spark.jars", ",".join(JARS))
             .config("spark.executor.memory", MEMORIA_EXECUTOR)
             .config("spark.executor.cores", CORES_EXECUTOR)
             .config("spark.executor.instances", INSTANCIAS_EXECUTOR)
             .config("spark.sql.shuffle.partitions", str(PARTICIONES_SHUFFLE))
             .getOrCreate())
    sc = spark.sparkContext

    print("Application ID:", sc.applicationId)
    print("Version de Spark:", spark.version, "| version de Scala:", sc._jvm.scala.util.Properties.versionNumberString())
    print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
    print("spark.eventLog.enabled:", sc.getConf().get("spark.eventLog.enabled", "no definido"))
    try:
        eventos(leer_kafka())
        print("Conector de Kafka para Structured Streaming: disponible")
    except Exception as error:
        print("ATENCION: el conector de Kafka no esta disponible ({}). Informar a la docente.".format(type(error).__name__))

    p, fs = _ruta(HDFS_T11)
    fs.mkdirs(p)
    print("Carpeta del taller en HDFS:", HDFS_T11, "(existe)" if existe(HDFS_T11) else "(NO SE PUDO CREAR)")
    print()

    subprocess.run([KB + "/kafka-topics.sh", "--bootstrap-server", BS, "--create", "--if-not-exists",
                    "--topic", TOPIC, "--partitions", str(PARTICIONES), "--replication-factor", str(REPLICACION)])
    time.sleep(2)
    print(subprocess.run([KB + "/kafka-topics.sh", "--bootstrap-server", BS, "--describe", "--topic", TOPIC],
                         capture_output=True, text=True).stdout)
    print("Semilla:", SEMILLA)

E1 | 2026-10-30 16:58:28 | sin sesion de Spark | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
Application ID: application_1793395800303_0001
Version de Spark: 3.5.7 | version de Scala: 2.12.18
spark.sql.shuffle.partitions: 4
spark.eventLog.enabled: true
Conector de Kafka para Structured Streaming: disponible
Carpeta del taller en HDFS: hdfs://hadoop-virtualbox:9000/user/hadoop/t11 (existe)

Created topic t11-lecturas.
Topic: t11-lecturas	TopicId: OUCzC1shSkS1R0yq_8Wd5w	PartitionCount: 3	ReplicationFactor: 2	Configs: min.insync.replicas=2,segment.bytes=536870912,retention.bytes=1073741824
	Topic: t11-lecturas	Partition: 0	Leader: 3	Replicas: 3,1	Isr: 3,1	Elr: 	LastKnownElr: 
	Topic: t11-lecturas	Partition: 1	Leader: 1	Replicas: 1,2	Isr: 1,2	Elr: 	LastKnownElr: 
	Topic: t11-lecturas	Partition: 2	Leader: 2	Replicas: 2,3	Isr: 2,3	Elr: 	LastKnownElr: 

Semilla: 5452


**Registrar en la hoja (fila E1):** Application ID · `ClusterId` · semilla · `spark.sql.shuffle.partitions` · versión de Scala · número de particiones del topic `t11-lecturas` (`PartitionCount`).

*Fuente:* salida de la celda anterior.

*No requiere captura.*

---
## Ejecución 2 — El mismo flujo, dos modos de salida

**Qué hace la celda** (unos dos minutos y medio):

1. Arranca **dos consultas idénticas salvo en el modo de salida**: `e2_completo`, en modo **Complete**, y `e2_anexar`, en modo **Append**. Ambas leen el topic, cuentan las lecturas por ventana *tumbling* de 30 segundos y por estación, con un watermark de 10 segundos, y escriben en una tabla en memoria del driver.
2. Espera a que las dos fijen su punto de partida en el topic. Con `startingOffsets = latest`, cada consulta empieza en el final del topic y solo lee lo que llegue después.
3. Lanza el productor en modo completo, unos 80 segundos, y muestra el avance cada 10 segundos.
4. Espera a que las dos consultas lean todo lo enviado, imprime el **RESUMEN** y detiene las consultas.

**Tiempo de evento y watermark.** Cada lectura trae su tiempo de evento (`ts_evento_ms`), el instante en que el sensor la tomó. Las ventanas se forman con ese tiempo, no con el de llegada. El watermark de cada lote es el máximo tiempo de evento observado hasta el lote anterior menos 10 segundos: Spark lo usa para decidir qué ventanas ya no pueden recibir datos.

**La ráfaga.** Hacia el segundo 30, el productor envía un grupo de lecturas cuyo tiempo de evento quedó unos dos minutos atrás: lecturas que llegan tarde. Para entonces el watermark ya avanzó mucho más allá de esa ventana.

**Mientras la celda corre**, abrir la pestaña Structured Streaming: las dos consultas aparecen en *Active Streaming Queries*. Conviene abrir la página de estadísticas de cada una y ver cómo se dibujan los gráficos.

In [11]:
encabezado("E2")
if not ya_ejecutada("E2"):
    for nombre in ("e2_completo", "e2_anexar"):
        borrar(HDFS_T11 + "/checkpoints/" + nombre)        # restos de un intento interrumpido
    lecturas = eventos(leer_kafka())
    q_completo = arrancar(conteo_por_ventana(lecturas), "e2_completo", "complete")
    q_anexar = arrancar(conteo_por_ventana(lecturas), "e2_anexar", "append")
    consultas_e2 = [q_completo, q_anexar]
    try:
        esperar_listas(consultas_e2)
        r_e2 = seguir(consultas_e2, lanzar_productor("completo", "E2"))
        informe_e2(consultas_e2, r_e2)
        marcar("E2")
    finally:
        print()
        detener(consultas_e2)

E2 | 2026-10-30 16:59:08 | application_1793395800303_0001 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
Consulta e2_completo iniciada (modo complete, salida memory).
Consulta e2_anexar iniciada (modo append, salida memory).


Consultas listas en 10 s: e2_completo, e2_anexar
Productor lanzado como proceso aparte del sistema operativo: PID 11927, modo completo.
  t =   0 s | productor enviando | e2_completo: lote 0, 0 filas leidas | e2_anexar: lote 0, 0 filas leidas


  t =  10 s | productor enviando | e2_completo: lote 1, 50 filas leidas | e2_anexar: lote 1, 56 filas leidas


  t =  20 s | productor enviando | e2_completo: lote 3, 154 filas leidas | e2_anexar: lote 3, 154 filas leidas


  t =  30 s | productor enviando | e2_completo: lote 5, 264 filas leidas | e2_anexar: lote 5, 264 filas leidas


  t =  40 s | productor enviando | e2_completo: lote 7, 382 filas leidas | e2_anexar: lote 7, 382 filas leidas


  t =  50 s | productor enviando | e2_completo: lote 9, 486 filas leidas | e2_anexar: lote 9, 486 filas leidas


  t =  60 s | productor enviando | e2_completo: lote 11, 593 filas leidas | e2_anexar: lote 11, 593 filas leidas


  t =  70 s | productor enviando | e2_completo: lote 13, 710 filas leidas | e2_anexar: lote 13, 710 filas leidas


  t =  76 s | productor terminado (codigo de salida 0)


  t =  93 s | lectura completa

E2 | RESUMEN | 2026-10-30 17:00:52 | application_1793395800303_0001 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
PRODUCTOR (kafka-python 2.2.21)
  Semilla: 5452   Inicio: 16:59:19   Fin: 17:00:35
  Lecturas a tiempo: 802   Lecturas de la rafaga: 16   Total enviado: 818   Errores de envio: 0
  Estaciones de la rafaga: 4 -> EST-01 (4), EST-05 (4), EST-07 (5), EST-08 (3)
  Tiempo de evento de la rafaga: tramo 16:57:30 - 16:57:40   (ventana tumbling 16:57:30 - 16:58:00)

LECTURA DE LAS CONSULTAS
  e2_completo  filas leidas:   818   coincide con lo enviado
  e2_anexar    filas leidas:   818   coincide con lo enviado

TABLAS EN MEMORIA, POR VENTANA            e2_completo          e2_anexar
  Ventana (inicio - fin)               estaciones lecturas  estaciones lecturas
  16:57:30 - 16:58:00  (*)                 4       16           -        -
  16:59:00 - 16:59:30                      8      115           8      115
  16:59:30 - 17:00:00                      8      319 

**Dónde mirar.** El RESUMEN tiene tres partes: lo que envió el productor, cuánto leyó cada consulta y las dos tablas en memoria, ventana por ventana. La ventana marcada con `(*)` es la que contiene el tiempo de evento de la ráfaga.

Después, en la pestaña Structured Streaming, la tabla *Completed Streaming Queries* muestra las dos consultas. Al hacer clic en el Run ID de cada una se abre su página de estadísticas. Interesan dos zonas:

- Arriba: **Input Rate**, **Process Rate**, **Input Rows** y **Batch Duration**, uno por lote.
- Más abajo: **Global Watermark Gap** (cuánto va el watermark por detrás del reloj de cada lote) y los gráficos de estado, entre ellos **Aggregated Number Of Total State Rows** y **Aggregated Number Of Rows Dropped By Watermark**.

Cada gráfico tiene dos partes: a la izquierda, la línea de tiempo, con un punto por lote; a la derecha, el histograma, que agrupa los lotes según su valor. Al pasar el cursor sobre un punto aparece un recuadro con el valor, el número de lote y la hora en que empezó ese lote, con milésimas de segundo; sobre una barra del histograma, cuántos lotes caen en ese intervalo. El símbolo (?) junto a cada título explica qué mide el gráfico.

En la pestaña **Jobs**, cada lote de cada consulta aparece como uno o más jobs. La columna *Job Id (Job Group)* muestra entre paréntesis el RunId de la consulta que lanzó el job: es el mismo mecanismo de grupos de jobs (`setJobGroup`) que los Talleres 7 y 8 usaron para etiquetar las ejecuciones.

**Registrar en la hoja (fila E2):**

- Del RESUMEN: total enviado · lecturas de la ráfaga · número de estaciones de la ráfaga y cuáles · ventana *tumbling* de la ráfaga · filas leídas por cada consulta · estaciones y lecturas de la ventana `(*)` en `e2_completo` · número de ventanas de cada tabla.
- De la tabla *Completed Streaming Queries*: los ocho primeros caracteres del Run ID y el *Latest Batch* de cada consulta.
- De la página de estadísticas de `e2_completo`, gráfico *Aggregated Number Of Total State Rows*: el valor del último punto.
- De la página de estadísticas de `e2_anexar`: en *Aggregated Number Of Total State Rows*, el valor del punto más alto y el del último punto · en *Aggregated Number Of Rows Dropped By Watermark*, el valor y la hora del punto más alto · en el histograma de *Global Watermark Gap*, el intervalo de la barra más larga, en segundos.

*Fuente:* captura C1 y páginas de estadísticas.

**Captura C1:** el bloque RESUMEN completo de esta celda.

**Captura C2:** la página de estadísticas de `e2_completo`, con los gráficos *Total State Rows* y *Rows Dropped By Watermark* y el recuadro del último punto de *Total State Rows*.

**Captura C3:** la página de estadísticas de `e2_anexar`, desde *Global Watermark Gap* hasta *Rows Dropped By Watermark*, con el recuadro del punto más alto de *Rows Dropped By Watermark*.

*Comparaciones que conviene tener preparadas:*

1. Las dos consultas leyeron el mismo topic al mismo tiempo. ¿Por qué cada una leyó todas las lecturas, en lugar de repartírselas como los consumidores de un grupo en el Taller 10?
2. ¿Qué hizo cada modo con la ráfaga? La tabla y el gráfico de descartes de cada consulta dan la respuesta.
3. El valor del punto más alto de *Rows Dropped By Watermark* no coincide con todas las cifras de la ráfaga del RESUMEN. ¿Con cuál coincide, y por qué con esa?
4. ¿Por qué *Total State Rows* solo crece en `e2_completo` y en cambio sube y baja en `e2_anexar`? ¿Y por qué la última ventana falta en la tabla de `e2_anexar`?

---
## Ejecución 3a — Ventanas deslizantes hacia HDFS, con checkpoint

**Qué hace la celda** (unos dos minutos): arranca una sola consulta, `e3_ventanas`, que cuenta las lecturas por estación en ventanas de 30 segundos que **se desplazan cada 10 segundos**, con el mismo watermark. Escribe el resultado en modo Append como archivos Parquet en HDFS y guarda su checkpoint, también en HDFS. Después lanza el productor, con la misma semilla y por lo tanto con la misma ráfaga, y al final detiene la consulta.

**Por qué Append.** El *sink* de archivos solo admite ese modo: cada ventana se escribe una sola vez, cuando el watermark la cierra.

**Qué guarda el checkpoint.** El resumen lista su contenido:

- `offsets/`: un archivo por lote, escrito **antes** de procesarlo, con el rango de offsets de Kafka que ese lote va a leer.
- `commits/`: un archivo por lote, escrito **después** de terminarlo.
- `state/`: el estado de la agregación, es decir, los conteos parciales de las ventanas que siguen abiertas.
- `sources/` y `metadata`: el punto de partida de la fuente y el identificador (Id) de la consulta.

La página de estadísticas de esta consulta **no** se captura en la Fase 1.

In [12]:
encabezado("E3a")
if not ya_ejecutada("E3a"):
    borrar(SALIDA_E3)                                   # restos de un intento interrumpido
    borrar(CHECKPOINT_E3)
    q_e3 = consulta_e3()
    try:
        esperar_listas([q_e3])
        r_e3a = seguir([q_e3], lanzar_productor("completo", "E3a"))
        informe_e3("E3a", q_e3, r_e3a)
        marcar("E3a")
    finally:
        print()
        detener([q_e3])

E3a | 2026-10-30 17:08:12 | application_1793395800303_0001 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
Consulta e3_ventanas iniciada (modo append, salida parquet).


Consultas listas en 6 s: e3_ventanas
Productor lanzado como proceso aparte del sistema operativo: PID 13246, modo completo.
  t =   0 s | productor enviando | e3_ventanas: lote 0, 0 filas leidas


  t =  10 s | productor enviando | e3_ventanas: lote 2, 73 filas leidas
  t =  20 s | productor enviando | e3_ventanas: lote 4, 186 filas leidas


  t =  30 s | productor enviando | e3_ventanas: lote 6, 300 filas leidas


  t =  40 s | productor enviando | e3_ventanas: lote 8, 425 filas leidas
  t =  50 s | productor enviando | e3_ventanas: lote 10, 527 filas leidas


  t =  60 s | productor enviando | e3_ventanas: lote 12, 638 filas leidas


  t =  70 s | productor enviando | e3_ventanas: lote 14, 748 filas leidas


  t =  75 s | productor terminado (codigo de salida 0)
  t =  93 s | lectura completa

E3a | RESUMEN | 2026-10-30 17:09:52 | application_1793395800303_0001 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
PRODUCTOR (kafka-python 2.2.21)
  Semilla: 5452   Inicio: 17:08:18   Fin: 17:09:33
  Lecturas a tiempo: 820   Lecturas de la rafaga: 16   Total enviado: 836   Errores de envio: 0
  Estaciones de la rafaga: 4 -> EST-01 (4), EST-05 (4), EST-07 (5), EST-08 (3)
  Tiempo de evento de la rafaga: tramo 17:06:30 - 17:06:40   (ventana tumbling 17:06:30 - 17:07:00)

LECTURA DE LAS CONSULTAS
  e3_ventanas  filas leidas:   836   coincide con lo enviado

CONSULTA e3_ventanas
  Id:    6a4858cb-6b07-4519-95cb-bedf25f7e4f5
  RunId: a58cef5b-2579-4c46-abb0-13df4bec9f94
  Lotes ejecutados en esta corrida: del 0 al 17

CHECKPOINT (hdfs://hadoop-virtualbox:9000/user/hadoop/t11/checkpoints/e3_ventanas)
  Contenido: commits, metadata, offsets, sources, state
  offsets/: 18 archivos (lotes 0 a 17)
  commits/: 18 archivo

**Registrar en la hoja (fila E3a):** los ocho primeros caracteres del Id y del RunId · lotes ejecutados en esta corrida (del … al …) · archivos en `offsets/` y en `commits/` · archivos `.parquet`, filas y ventanas distintas de la salida.

*Fuente:* RESUMEN de la celda.

*No requiere captura.*

---
## Ejecución 3b — Reiniciar desde el checkpoint

La consulta `e3_ventanas` quedó detenida al final de E3a. La celda la arranca de nuevo **con la misma definición y el mismo checkpoint**, sin borrar nada, y lanza el productor en modo corto: 20 segundos de lecturas a tiempo, sin ráfaga.

Conviene comparar con la fila E3a de la hoja: el Id, el RunId, la numeración de los lotes, los archivos del checkpoint y las filas leídas.

In [13]:
encabezado("E3b")
if not os.path.exists(os.path.join(T11, "E3a.hecho")):
    print("ATENCION: la Ejecucion 3a no se ha completado. Ejecutarla antes de esta celda.")
elif not ya_ejecutada("E3b"):
    with open(os.path.join(T11, "E3a_consulta.json")) as f:
        antes = json.load(f)
    print("Ultimo lote confirmado antes de detener la consulta (E3a):", antes["ultimo_commit"])
    print("Se reinicia la misma consulta, con el mismo checkpoint. No se borra nada.")
    q_e3 = consulta_e3()
    try:
        esperar_listas([q_e3])
        r_e3b = seguir([q_e3], lanzar_productor("corto", "E3b"))
        informe_e3("E3b", q_e3, r_e3b)
        marcar("E3b")
    finally:
        print()
        detener([q_e3])

E3b | 2026-10-30 17:10:16 | application_1793395800303_0001 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
Ultimo lote confirmado antes de detener la consulta (E3a): 17
Se reinicia la misma consulta, con el mismo checkpoint. No se borra nada.
Consulta e3_ventanas iniciada (modo append, salida parquet).
Consultas listas en 4 s: e3_ventanas
Productor lanzado como proceso aparte del sistema operativo: PID 13787, modo corto.
  t =   0 s | productor enviando | e3_ventanas: lote 18, 0 filas leidas


  t =  10 s | productor enviando | e3_ventanas: lote 18, 51 filas leidas
  t =  20 s | productor enviando | e3_ventanas: lote 20, 161 filas leidas
  t =  21 s | productor terminado (codigo de salida 0)


  t =  37 s | lectura completa

E3b | RESUMEN | 2026-10-30 17:10:57 | application_1793395800303_0001 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
PRODUCTOR (kafka-python 2.2.21)
  Semilla: 5452   Inicio: 17:10:20   Fin: 17:10:40
  Lecturas a tiempo: 217   Lecturas de la rafaga: 0   Total enviado: 217   Errores de envio: 0

LECTURA DE LAS CONSULTAS
  e3_ventanas  filas leidas:   217   coincide con lo enviado

CONSULTA e3_ventanas
  Id:    6a4858cb-6b07-4519-95cb-bedf25f7e4f5
  RunId: 0553ffbd-652c-4b50-b051-02c6764fc950
  Lotes ejecutados en esta corrida: del 18 al 22

CHECKPOINT (hdfs://hadoop-virtualbox:9000/user/hadoop/t11/checkpoints/e3_ventanas)
  Contenido: commits, metadata, offsets, sources, state
  offsets/: 23 archivos (lotes 0 a 22)
  commits/: 23 archivos (lotes 0 a 22)

SALIDA PARQUET (hdfs://hadoop-virtualbox:9000/user/hadoop/t11/salida/e3_ventanas)
  Archivos .parquet: 46   Filas: 96   Ventanas distintas: 12


Consulta e3_ventanas detenida.


**Registrar en la hoja (fila E3b):** los ocho primeros caracteres del Id y del RunId · último lote confirmado en E3a · lotes ejecutados en esta corrida (del … al …) · archivos en `offsets/` y en `commits/` · filas leídas · *Latest Batch* de las dos filas de `e3_ventanas` en la tabla *Completed Streaming Queries*.

*Fuente:* RESUMEN de la celda y página principal de la pestaña Structured Streaming.

**Captura C4:** la tabla *Completed Streaming Queries*, con sus cuatro filas: `e2_completo`, `e2_anexar` y dos de `e3_ventanas`. Deben verse las columnas *Name*, *ID*, *Run ID*, *Start Time* y *Latest Batch*.

*Comparación que conviene tener preparada:* la segunda fila de `e3_ventanas` conserva un dato de la primera y cambia otro. ¿Cuál conserva, cuál cambia, y dónde estaba guardado lo que le permitió continuar? ¿Qué habría ocurrido con un checkpoint vacío?

---
## Cierre de la Fase 1

La celda siguiente detiene cualquier consulta que haya quedado activa. **No detiene la aplicación de Spark.**

Antes del corte, verificar que:

1. La hoja de registro está completa, filas E1 a E3b.
2. El archivo de Word contiene las cuatro capturas, numeradas C1 a C4.
3. El archivo se convirtió a PDF y se entregó por Microsoft Teams.

**Durante la Fase 2:**

- **No ejecutar `spark.stop()`** ni apagar el kernel. La aplicación debe seguir viva: una pregunta exige una captura nueva del portal.
- **Cerrar la pestaña de este notebook.** El kernel sigue activo en el servidor y el portal continúa respondiendo.
- En pantalla solo puede estar el portal: YARN y la Spark UI de la aplicación propia. La terminal no se usa.
- Si el portal deja de responder, informar a la docente: la aplicación también puede consultarse en el History Server (`localhost:18080`).
- El cuestionario se responde en **Microsoft Forms**. La captura nueva se carga en el mismo formulario.

**Material permitido en la Fase 2:** el portal de Spark de la aplicación propia, únicamente para consulta; la hoja de registro diligenciada; el archivo de capturas entregado en el corte, únicamente para consulta; los notebooks de sesiones anteriores en la versión publicada por la docente en Microsoft Teams; los apuntes propios manuscritos.

**No permitido:** herramientas de inteligencia artificial; la terminal; copias propias o editadas de los notebooks; apuntes digitales; el notebook de este taller, que debe permanecer cerrado.

In [ ]:
encabezado("Cierre de la Fase 1")
activas = spark.streams.active
for q in activas:
    q.stop()
print("Consultas que seguian activas y se detuvieron:", len(activas))
print("La aplicacion de Spark sigue viva:", spark.sparkContext.applicationId)
print("No ejecutar spark.stop(): el portal debe seguir disponible durante la Fase 2.")

In [ ]:
# ATENCION: estas lineas deben permanecer comentadas hasta que la docente lo indique, al terminar la Fase 2.
# Ejecutarlas cierra la aplicacion de Spark y deja el portal inaccesible.
# spark.stop()
# !bash /home/hadoop/kafka_cluster.sh detener

---
## Solo con autorización de la docente

Si una ejecución se interrumpió o falló, la docente puede autorizar repetirla. La celda siguiente elimina la marca que lo impide; después se ejecuta de nuevo la celda de esa ejecución, sin repetir las demás. No es necesario borrar carpetas ni reiniciar servicios: la propia celda limpia sus checkpoints y su salida. Al repetirla cambian las cifras de esa ejecución, pero no la ráfaga, que depende de la semilla. Se rehacen la fila de la hoja y las capturas de esa ejecución; las demás se conservan. Las consultas del intento anterior siguen en *Completed Streaming Queries*: las capturas se toman de la corrida con la *Start Time* más reciente. Repetir E3a obliga a repetir también E3b.

In [ ]:
# Solo con autorizacion de la docente, para repetir una ejecucion que fallo.
# Quitar el # de la linea que corresponda y ejecutar la celda.
# habilitar_repeticion("E2")
# habilitar_repeticion("E3a")   # repetir E3a obliga a repetir tambien E3b
# habilitar_repeticion("E3b")